# Extended desirability calibration -- protein, dependency, copy number

**Why we're doing this.** `scoring/resources/desirability_constants.json` (built by `scoring/build_desirability_constants.py`) only covers RNA. No source paper gives a calibration procedure for the other three continuous layers scoring/desirability.py needs -- protein, CRISPR dependency, and copy number -- so this extends the same percentile method to all three.

**The computation lives in `scoring/build_extended_desirability_constants.py`, not in this notebook.** That script is what actually ships the constants (`python scoring/build_extended_desirability_constants.py`, run by hand from the repo root -- or `python scoring/build_calibration_constants.py` to rebuild every calibration file in one command). This notebook imports the very same functions and walks through them step by step, so the audit you are reading and the file the scorer loads can never drift apart.

**Output goes to a separate, new file** -- `scoring/resources/desirability_constants_extended.json` -- never merged into the RNA constants file, so the two calibration passes stay independently reproducible and independently auditable.

**Three per-layer design decisions, each stated so it can be challenged:**

1. **Protein gets one pooled global `(L, T)`, not a per-gene pair.** Protein z-scores already arrive standardized per-protein upstream, so a per-gene percentile would be near-circular, and coverage per gene is too sparse (median far below the >=30-line gate below) for a stable per-gene estimate. We pool across every `detected=True` measurement instead.
2. **Dependency and copy number both get per-gene pairs, gated at >= 30 measured lines/gene** -- enough lines that a 10th/90th percentile is a real estimate, not noise from a handful of profiles.
3. **Dependency is calibrated on `-dependency_score` ("dependency strength"), not the raw Chronos value.** Chronos's own convention is that a more negative score means a stronger dependency -- the opposite of "higher is more desirable", the direction RNA/protein/copy-number all share. Negating up front means `desirability_transform`'s shared higher-is-better logic stays correct everywhere, including here.

**P5 note on protein.** `preprocessing/impute.py` now fills the `zscore` of not-detected rows with a detection-floor value *before* this notebook ever runs (v4 expected this fill to happen downstream, inside scoring). `detected` is untouched by that fill and stays the ground truth for measured vs. measured_absent -- so filtering on `detected == True` below still correctly excludes every imputed row from the percentile, exactly as it always was meant to.

In [ ]:
import json
import sys
import time

sys.path.insert(0, "..")  # this notebook runs from scoring/notebooks/; the builder is one level up

import build_extended_desirability_constants as build

DATA_DIR = "../../data/processed"
RESOURCES_DIR = "../resources"

MIN_N = build.MIN_N              # 30 -- the production per-gene calibration gate
MIN_N_FLOOR = build.MIN_N_FLOOR  # 10 -- below V6-3's swept range, SENSITIVITY.md SS3
print(f"production MIN_N={MIN_N}, sweep floor={MIN_N_FLOOR}")

## 1. Protein -- pooled global (L, T), detected rows only

In [ ]:
t0 = time.time()
protein_L, protein_T, n_detected = build.compute_protein_lt(DATA_DIR)
print(f"n_detected={n_detected:,}, L={protein_L:.4f}, T={protein_T:.4f} ({time.time() - t0:.1f}s)")

## 2. Dependency -- per-gene (L, T) on dependency strength (-dependency_score), n >= 30

In [ ]:
t1 = time.time()
dependency = build.load_dependency_strength(DATA_DIR)
dep_q, dep_counts = build.compute_gene_lt(dependency, "strength")
dep_kept = build.apply_count_floor(build.apply_discrimination_gate(dep_q), dep_counts, MIN_N)
print(f"dependency: kept {len(dep_kept):,} of {len(dep_q):,} genes ({time.time() - t1:.1f}s)")
del dependency

## 3. Copy number -- per-gene (L, T) on the continuous value, n >= 30

In [ ]:
t1 = time.time()
copy_number = build.load_copy_number(DATA_DIR)
cn_q, cn_counts = build.compute_gene_lt(copy_number, "copy_number")
cn_kept = build.apply_count_floor(build.apply_discrimination_gate(cn_q), cn_counts, MIN_N)
print(f"copy_number: kept {len(cn_kept):,} of {len(cn_q):,} genes ({time.time() - t1:.1f}s)")
del copy_number

## 4. Write `scoring/resources/desirability_constants_extended.json`

In [ ]:
constants = build.build_production_constants(protein_L, protein_T, n_detected, dep_q, dep_kept, cn_q, cn_kept)

out_path = f"{RESOURCES_DIR}/desirability_constants_extended.json"
with open(out_path, "w") as f:
    json.dump(constants, f, indent=2)
print(f"Wrote {out_path}")

## 5. V6-3 sweep-floor export -- `scoring/resources/desirability_constants_extended_sweep_floor.json`

Same rationale as `calibration_rna_eda.ipynb` §7: `docs/plan/PARAMETERS.md` §11 requires sweeping
`min_calibration_n` (default 30, row 9, tag T2) as low as 10 (`docs/plan/SENSITIVITY.md` §3's `[10,
100]` range). `dep_q`/`cn_q` above already hold every gene's percentiles before any count floor is
applied, so no new statistics are needed -- only a lower cut, plus persisting the per-gene counts
(`dependency_gene_counts`, `copy_number_gene_counts`) the production file does not carry. Protein has
no per-gene N-gate (it is one pooled pair, §1) so nothing to re-cut there.

In [ ]:
dep_floor = build.apply_count_floor(build.apply_discrimination_gate(dep_q), dep_counts, MIN_N_FLOOR)
cn_floor = build.apply_count_floor(build.apply_discrimination_gate(cn_q), cn_counts, MIN_N_FLOOR)
print(f"dependency: {len(dep_floor):,} genes at floor={MIN_N_FLOOR} vs. {len(dep_kept):,} at production MIN_N={MIN_N}")
print(f"copy_number: {len(cn_floor):,} genes at floor={MIN_N_FLOOR} vs. {len(cn_kept):,} at production MIN_N={MIN_N}")

sweep_floor_constants = build.build_sweep_floor_constants(dep_q, dep_floor, dep_counts, cn_q, cn_floor, cn_counts)

floor_out_path = f"{RESOURCES_DIR}/desirability_constants_extended_sweep_floor.json"
with open(floor_out_path, "w") as f:
    json.dump(sweep_floor_constants, f, indent=2)
print(f"Wrote {floor_out_path}")